# Chapter 3: Cross-Encoder Re-ranking and Context Compression

Retrieving too many documents can clutter the LLM context, diluting its focus and causing performance degradation (known as **'Lost in the Middle'**).

This notebook implements:
1. **Cross-Encoder Re-ranking**: Evaluates retrieved chunks by feeding the query and document together through an attention layer. This yields far higher precision than standard vector search (bi-encoders).
2. **Context Compression**: Splitting the top re-ranked documents into individual sentences, and filtering out sentences that contain zero semantic overlap with the search intent. This strips away boilerplate and saves token costs.

In [ ]:
import os
import sys
sys.path.append(os.path.dirname(os.getcwd()))

from src.ranking.reranker import CrossEncoderReranker, ContextCompressor

reranker = CrossEncoderReranker()
compressor = ContextCompressor(sentences_limit=3)

### Step 1: Re-ranking a candidate document list

In [ ]:
query = "How is encryption handled on S3?"
mock_docs = [
    {"id": "doc1", "text": "S3 buckets enforce KMS encryption using customer-managed keys rotated annually. AWS IAM roles enforce least privilege.", "metadata": {"source": "sec.md"}},
    {"id": "doc2", "text": "All staff members must complete security awareness training within 30 days of hiring. Okta SSO is federated.", "metadata": {"source": "sec.md"}},
    {"id": "doc3", "text": "Buckets are configured with default SSE-KMS headers. Bucket policies actively block raw unencrypted uploads.", "metadata": {"source": "sec.md"}}
]

reranked_docs = reranker.rerank(query, mock_docs, top_n=2)
print("Reranked Documents:")
for doc in reranked_docs:
    print(f"- Score: {doc['rerank_score']:.4f} | ID: {doc['id']} | Text: {doc['text'][:120]}...")

### Step 2: Context Compression
Compressing document paragraphs down to their primary responsive sentences.

In [ ]:
original_text = "Acme Corp S3 storage buckets are secure. We enforce SSE-KMS encryption. Unused keys are rotated. IAM credentials expire in 8 hours. mTLS Istio Mesh is activated for microservice systems. Let's maintain high network reliability."
compressed = compressor.compress("KMS encryption keys", original_text)

print(f"Original Length: {len(original_text)} chars")
print(f"Compressed Length: {len(compressed)} chars")
print(f"\nCompressed Text payload:\n'{compressed}'")